# PBAIH11-B 實作二：從特徵表到可分析的矩陣

**課堂安排（主流程）：Part 1–3 在上課第 2 節後段、Part 4–7 在第 3 節；沒做完的部分課後接著做**（這裡的 **Part** 是 notebook 的章節，「第 2 節」「第 3 節」指上課的第幾節）｜對應投影片步驟 ⑤ QC 與過濾 → ⑥ 缺值 → ⑦ 校正與正規化 → ⑧ 轉換與縮放

### 這份資料是什麼？（先老實說）
- 資料：MetaboLights **MTBLS79** 的 `Dataset07__SFPM.xlsx`（Kirwan 等人 2014，*Sci Data* 1:140012）。
- 樣本：**牛（C）與羊（S）的心臟組織萃取物**，共 20 個生物樣本（各 10 個），在 **8 個批次、7 天**內**重複測量**，並穿插 pooled QC 樣本。
- 儀器：**直接注入質譜（DIMS）**——樣本不經層析管柱，**沒有保留時間（RT）**，所以每個特徵只是一個 m/z 峰，而不是實作一的 `m/z@RT`。
- 但**「拿到特徵表之後」的前處理步驟（QC 過濾、缺值、漂移與批次校正、正規化、轉換縮放）與 LC-MS 完全相同**，這正是今天要練的部分。
- 這份資料的設計很適合教學：同一個生物樣本在不同批次被量了好幾次，所以我們可以**用「同一樣本的重複測量是否變一致」獨立檢查校正效果**，不必只相信 QC。

### 今天的路線
| 主流程（必做） | 延伸／教師示範（執行現成程式並解讀） |
|---|---|
| Part 0 環境與資料 → Part 1 讀入與檢視、處理前 PCA → Part 2 QC RSD 過濾 → Part 3 缺值與 KNN 補值 → Part 4 QC-LOESS 漂移校正 → Part 5 PQN → Part 6 log2＋Pareto → Part 7 處理後 PCA 與指標 → Part 8 總結 | 3E 補值法比較（遮蔽再還原）、4E 批次校正：簡化版位置／尺度 vs ComBat |

練習題共 5 題（✏️ 標記），都是「補完這一行」。看到 `____` 就是要你填的地方。

> 🆘 **卡住時**：① 先讀錯誤訊息最後一行（哪個變數、哪一行）→ ② 問 Colab 的 Gemini（把錯誤訊息貼給它，請它「解釋」而不是「直接改」）→ ③ 打開下面的提示。

## Part 0. 環境與資料

### 執行環境
- 在 **Google Colab** 執行即可；本 notebook 只用 Colab 內建套件：pandas、numpy、scipy、scikit-learn、statsmodels、matplotlib、seaborn、openpyxl（讀 xlsx）。
- 開發時驗證的版本（= 2026 年秋 Colab 內建版本）：pandas 2.2.3、scipy 1.16.3、scikit-learn 1.6.1、statsmodels 0.15.0、matplotlib 3.10.0、seaborn 0.13.2。
- **唯一需要另外安裝的**：延伸 4E 的 `combat==0.3.3`（ComBat 的 Python 實作，只多裝 `mpmath`），在那一節才用 `%pip` 安裝；主流程不需要。
- 圖表標籤用英文（Colab 預設沒有中文字型）。

In [ ]:
import sys, time
import numpy as np
import pandas as pd
import scipy, sklearn, statsmodels, matplotlib, seaborn as sns
import matplotlib.pyplot as plt

print("Python      ", sys.version.split()[0])
for mod in [pd, np, scipy, sklearn, statsmodels, matplotlib, sns]:
    print(f"{mod.__name__:12s}", mod.__version__)

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 120)

### 下載資料
- 課程版：從課程的 GitHub 下載（約 4 MB）。
- 備援：GitHub 失敗時，直接從 MetaboLights 公開 FTP（HTTPS）下載同一個檔案。
- 下載後用 MD5 校驗碼確認檔案完整。

In [ ]:
import os, hashlib, urllib.request
from pathlib import Path

DATA_FILE = "Dataset07__SFPM.xlsx"
GITHUB_URL = ("https://raw.githubusercontent.com/wang3yuan/PBAIH/main/"
              "data/PBAIH11/Dataset07__SFPM.xlsx")   # 課程版
FTP_URL = ("https://ftp.ebi.ac.uk/pub/databases/metabolights/studies/public/"
           "MTBLS79/FILES/Dataset07__SFPM.xlsx")  # 備援：MetaboLights 公開檔案
EXPECTED_MD5 = "30413b355a2163effb221eca8857ec2c"

if os.environ.get("PBAIH11_DATA_DIR"):           # 開發驗證用：本機資料夾
    data_path = Path(os.environ["PBAIH11_DATA_DIR"]) / DATA_FILE
else:
    data_path = Path(DATA_FILE)
    if not data_path.exists():
        try:
            urllib.request.urlretrieve(GITHUB_URL, data_path)
        except Exception as e:                       # GitHub 失敗就改用 MetaboLights
            print("GitHub 下載失敗（", e, "），改從 MetaboLights 下載 ...")
            urllib.request.urlretrieve(FTP_URL, data_path)

md5 = hashlib.md5(data_path.read_bytes()).hexdigest()
print(data_path, f"{data_path.stat().st_size/1e6:.1f} MB", "MD5 OK" if md5 == EXPECTED_MD5 else f"MD5 不符：{md5}")

## Part 1. 讀入與檢視

**為什麼**：前處理的每一步都要用到「樣本中繼資料」（誰是 QC、哪個批次、第幾個注入）。先確認資料長什麼樣子、有沒有怪東西，比急著跑演算法重要。

**怎麼做**：xlsx 有三個工作表：`data`（樣本 × m/z 峰的強度）、`meta`（批次、類別）、`peak`（峰的編號與 m/z）。

In [ ]:
raw = pd.read_excel(data_path, sheet_name="data", index_col=0)
meta = pd.read_excel(data_path, sheet_name="meta")
peaks = pd.read_excel(data_path, sheet_name="peak")
print("data:", raw.shape, " meta:", meta.shape, " peak:", peaks.shape)
display(raw.iloc[[0, 1, -2, -1], :5])   # 注意最後一列！
display(meta.head())

**資料怪癖（真實資料常見）**
1. `data` 最後一列叫 `mean intensity`，是各峰的平均，**不是樣本**，要刪掉。
2. `meta` 只有前 4 欄有用；右邊有一格零散資料（第 23–26 欄），忽略。
3. **檔案沒有注入序欄位**。樣本名稱 `QC01…QC39` 的編號與列順序一致，所以我們**假設列順序＝注入序**（QC35 不在檔案裡）。這是一個假設，正式分析要向資料提供者確認。
4. 欄名是 m/z 數字；我們改成 `mz70.03364` 這種字串，方便取用。

In [ ]:
raw = raw.drop(index="mean intensity")
meta = meta.iloc[:len(raw), :4].copy()
meta.index = raw.index
meta["Batch"] = meta["Batch"].astype(int)
meta["Order"] = np.arange(1, len(meta) + 1)        # 假設：列順序 = 注入序
raw.columns = [f"mz{c:.5f}" for c in raw.columns]

# 檢查：樣本名稱（去掉 batchXX_ 與前導 0）要和 meta 的 Class2 對得上
name = raw.index.str.replace(r"^[Bb]atch\d+_", "", regex=True).str.replace(r"([A-Z]+)0*(\d+)", r"\1\2", regex=True)
assert (name == meta["Class2"]).all()

is_qc = (meta["Class"] == "QC").values
print(f"{raw.shape[0]} 個樣本 × {raw.shape[1]} 個 m/z 峰")
print(meta["Class"].value_counts().to_dict())

### ✏️ 練習 1：批次和類別有沒有混淆？
如果某個批次只有牛、另一個批次只有羊，那「批次差異」和「物種差異」就分不開了（**混淆，confounding**）——這時任何事後的批次校正都可能把真正的生物差異一起校正掉。請用 `pd.crosstab` 做「批次 × 類別」的交叉表。

In [ ]:
design = pd.crosstab(____, ____)   # TODO：列 = 批次、欄 = 類別（都在 meta 裡）
design

<details><summary>💡 提示（先自己試，再打開）</summary>

`pd.crosstab(列的 Series, 欄的 Series)`；兩個欄位名稱分別是 `"Batch"` 與 `"Class"`。

</details>

> 🆘 **卡住時**：① 先讀錯誤訊息最後一行（哪個變數、哪一行）→ ② 問 Colab 的 Gemini（把錯誤訊息貼給它，請它「解釋」而不是「直接改」）→ ③ 打開下面的提示。

In [ ]:
# 同一個生物樣本被量了幾次？（C1–C10、S1–S10 各在不同批次重複測量）
reps = meta.loc[~is_qc, "Class2"].value_counts()
print("每個生物樣本的重複測量次數：", reps.min(), "到", reps.max(), "次；共", reps.size, "個生物樣本")

# 注入序列一覽：每一點是一次注入
fig, ax = plt.subplots(figsize=(12, 2.2))
colors = {"C": "#0F8B8D", "S": "#6C5CE7", "QC": "#EE6C4D"}
for cl, c in colors.items():
    m = meta["Class"] == cl
    ax.scatter(meta.loc[m, "Order"], meta.loc[m, "Batch"], c=c, s=60 if cl == "QC" else 25, label=cl)
ax.set_xlabel("Injection order"); ax.set_ylabel("Batch"); ax.legend(ncol=3, loc="upper left")
ax.set_title("Injection sequence (QC interleaved within each batch)")
plt.show()

**看結果**：每個批次都同時有牛、羊和 QC，數量大致平衡 → **批次與類別沒有混淆**（這是好的實驗設計，請記住它不是理所當然）。每批次只有 4–7 個 QC，這會限制後面 LOESS 能多「彎」。

### 處理前的 PCA
先看看原始資料的整體結構。為了能做 PCA，暫時把 0 用「該峰最小非零值的一半」填上，再取 log2（第 3、6 節會正式處理）。下面先定義幾個整份 notebook 會用到的小工具：
- `run_pca`：mean-centering ＋ Pareto scaling（Part 6 解釋）後做 PCA。
- `pca_metrics`：量化指標——**QC 離散度**（QC 到 QC 質心的平均距離 ÷ 全部樣本到總質心的平均距離，越小越聚攏）、**重複測量離散度**（同一生物樣本的重複測量到自己質心的距離，同樣除以總距離）、**批次對 PC 的 R²**、**類別對 PC1 的 R²**（R² = 組間平方和 ÷ 總平方和）。

In [ ]:
from sklearn.decomposition import PCA

BATCH_COLORS = ["#0F8B8D", "#8D99AE", "#F2B134", "#6C5CE7", "#2E86AB", "#A23B72", "#1A2238", "#C1666B"]

def run_pca(L, n=2):
    """L：log 尺度、沒有缺值的矩陣（樣本 × 特徵）。回傳 PC 分數與解釋變異比例。"""
    Z = (L - L.mean()) / np.sqrt(L.std())          # mean-centering + Pareto scaling
    pca = PCA(n_components=n).fit(Z.values)
    scores = pd.DataFrame(pca.transform(Z.values), index=L.index, columns=["PC1", "PC2"])
    return scores, pca.explained_variance_ratio_

def r2_groups(y, groups):
    """one-way ANOVA 的 R²（= eta squared）：組間平方和 / 總平方和"""
    y = np.asarray(y, float); g = np.asarray(groups); m = y.mean()
    between = sum(((y[g == k].mean() - m) ** 2) * (g == k).sum() for k in np.unique(g))
    return between / ((y - m) ** 2).sum()

def pca_metrics(scores):
    S = scores.values
    total = np.linalg.norm(S - S.mean(0), axis=1).mean()
    qc = S[is_qc]
    qc_disp = np.linalg.norm(qc - qc.mean(0), axis=1).mean() / total
    bio = scores[~is_qc]; ind = meta.loc[~is_qc, "Class2"]
    rep_disp = np.mean([np.linalg.norm(g.values - g.values.mean(0), axis=1).mean()
                        for _, g in bio.groupby(ind)]) / total
    return pd.Series({
        "QC dispersion": qc_disp,
        "Replicate dispersion": rep_disp,
        "R2 batch~PC1": r2_groups(S[:, 0], meta["Batch"]),
        "R2 batch~PC2": r2_groups(S[:, 1], meta["Batch"]),
        "R2 class~PC1 (C vs S)": r2_groups(bio["PC1"], meta.loc[~is_qc, "Class"]),
    })

def plot_pca(scores, var, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for b in sorted(meta["Batch"].unique()):
        m = (meta["Batch"] == b).values & ~is_qc
        axes[0].scatter(scores.loc[m, "PC1"], scores.loc[m, "PC2"], s=25, color=BATCH_COLORS[b-1], label=f"batch {b}")
    axes[0].scatter(scores.loc[is_qc, "PC1"], scores.loc[is_qc, "PC2"], s=90, marker="*",
                    color="#EE6C4D", edgecolor="k", label="QC")
    for cl, c in colors.items():
        m = (meta["Class"] == cl).values
        axes[1].scatter(scores.loc[m, "PC1"], scores.loc[m, "PC2"], s=90 if cl == "QC" else 25,
                        marker="*" if cl == "QC" else "o", color=c, label=cl)
    for ax, t in zip(axes, ["colored by batch", "colored by class"]):
        ax.set_xlabel(f"PC1 ({var[0]:.0%})"); ax.set_ylabel(f"PC2 ({var[1]:.0%})")
        ax.set_title(f"{title}: {t}"); ax.legend(fontsize=8, ncol=2)
    plt.tight_layout(); plt.show()

X_all = raw.replace(0, np.nan)                       # 0 視為缺值（Part 3 說明）
L_before = np.log2(X_all.fillna(X_all.min() / 2))    # 暫時的簡單填補，只為了畫 PCA
scores_before, var_before = run_pca(L_before)
plot_pca(scores_before, var_before, "Before")
metrics_before = pca_metrics(scores_before)
metrics_before.round(3)

**看結果**：PC1 幾乎完全由物種（牛 vs 羊）決定；PC2 則被批次主導（批次 7–8 和其他批次分開），QC 也沒有聚在一起。

**想一想**：如果今天的研究問題是「牛和羊哪裡不一樣」，批次效應好像不太影響 PC1。那為什麼還要校正？（提示：真實研究中的組間差異通常比「不同物種」小得多。）

## Part 2. QC 的 RSD 分布與過濾（投影片 ⑤）

**為什麼**：pooled QC 是同一管混合樣本反覆注入，理論上每次都該一樣。某個峰在 QC 之間變動很大，就代表它的量測不可靠，生物差異也就看不出來。

**怎麼做**：每個峰計算 QC 的相對標準差 RSD = 標準差 ÷ 平均 × 100%，超過門檻的峰刪掉。
- 門檻 **20–30% 是常見的經驗值**，不是定論（Broadhurst 2018 的 QC 指引有討論）；這裡寫成參數 `RSD_MAX`，你可以改。
- 0 在這份資料代表「沒偵測到」，計算 RSD 時先當成缺值（不納入）。

### ✏️ 練習 2：計算每個峰在 QC 樣本的 RSD（%）

In [ ]:
RSD_MAX = 30          # 門檻（%），經驗值，可調

qc = X_all[is_qc]     # 只取 QC 樣本（0 已視為缺值）
rsd_qc = ____         # TODO：每一欄（每個峰）的 標準差 ÷ 平均 × 100

<details><summary>💡 提示（先自己試，再打開）</summary>

DataFrame 的 `.std(ddof=1)` 與 `.mean()` 預設就是「每一欄」計算；兩者相除再乘 100。

</details>

> 🆘 **卡住時**：① 先讀錯誤訊息最後一行（哪個變數、哪一行）→ ② 問 Colab 的 Gemini（把錯誤訊息貼給它，請它「解釋」而不是「直接改」）→ ③ 打開下面的提示。

In [ ]:
keep_rsd = rsd_qc <= RSD_MAX
print(f"QC RSD 中位數：{rsd_qc.median():.1f}%")
for t in [20, 30, 40]:
    print(f"  門檻 {t}%：保留 {(rsd_qc <= t).sum()} / {len(rsd_qc)} 個峰")

# 對照：如果只看「同一批次內」的 QC，RSD 是多少？
rsd_within = qc.groupby(meta.loc[is_qc, "Batch"]).apply(lambda d: d.std(ddof=1) / d.mean() * 100)
print(f"批次內 QC RSD 中位數（各批次再取中位數）：{rsd_within.median(axis=1).median():.1f}%")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(rsd_qc.clip(upper=100), bins=60, color="#0F8B8D")
ax.axvline(RSD_MAX, color="#EE6C4D", lw=2, label=f"threshold {RSD_MAX}%")
ax.set_xlabel("QC RSD (%) (clipped at 100)"); ax.set_ylabel("Number of peaks"); ax.legend()
plt.show()

X_f = X_all.loc[:, keep_rsd]
print("過濾後：", X_f.shape)

**想一想**
1. 跨批次的 QC RSD（中位數約 24%）遠大於批次內（約 10%）。差距來自哪裡？
2. 那麼「先用跨批次 RSD 過濾、再做批次校正」，會不會冤枉一些其實很穩、只是被批次效應拖累的峰？（Part 7 的敏感度分析會試「先校正再過濾」。）

## Part 3. 缺值：從哪裡來、怎麼補（投影片 ⑥）

**為什麼**：補值方法要配合缺值機制。
- **MNAR（左截尾）**：濃度低於偵測極限 → 缺值集中在低強度的峰，適合用「最小值類」補值。
- **MAR／MCAR**：演算法沒抓到、隨機掉值 → 適合 KNN、random forest 等利用其他樣本或特徵的方法。
- Kokla 2019 比較 9 種方法的結論正是如此；Do 2018 在 KORA 血清資料則發現缺值主要和「依分析日而異的偵測極限」有關。

**怎麼做**：先看每個峰的缺值比例，再看「缺值比例 vs 平均強度」和「缺值集中在哪些批次」。

### ✏️ 練習 3：計算每個峰的缺值比例

In [ ]:
miss_all = ____      # TODO：每一欄（每個峰）缺值的比例（0～1）

<details><summary>💡 提示（先自己試，再打開）</summary>

`X_all.isna()` 會得到 True/False 表；True 當成 1，對每一欄取平均就是比例。

</details>

> 🆘 **卡住時**：① 先讀錯誤訊息最後一行（哪個變數、哪一行）→ ② 問 Colab 的 Gemini（把錯誤訊息貼給它，請它「解釋」而不是「直接改」）→ ③ 打開下面的提示。

In [ ]:
from scipy.stats import spearmanr

print(f"整體缺值（0）比例：{X_all.isna().values.mean():.1%}；有缺值的峰：{(miss_all > 0).sum()} / {len(miss_all)}")
print(f"單一峰最大缺值比例：{miss_all.max():.1%}（推測原作者已先做過偵測率過濾）")

log_mean = np.log2(X_all.mean())
has_miss = miss_all > 0
rho, p = spearmanr(miss_all[has_miss], log_mean[has_miss])
print(f"有缺值的峰：缺值比例 vs log2 平均強度 Spearman rho = {rho:.2f}（p = {p:.1e}）")

miss_by_batch = X_all.isna().groupby(meta["Batch"]).mean().mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(log_mean, miss_all * 100, s=6, alpha=0.4, color="#0F8B8D")
axes[0].set_xlabel("log2 mean intensity"); axes[0].set_ylabel("Missing (%)")
axes[0].set_title("Missing rate vs intensity (each dot = one peak)")
axes[1].bar(miss_by_batch.index, miss_by_batch.values * 100, color="#8D99AE")
axes[1].set_xlabel("Batch"); axes[1].set_ylabel("Missing values (%)"); axes[1].set_title("Missing by batch")
plt.tight_layout(); plt.show()

**看結果**：缺值比例和強度只有**弱的**負相關（低強度的峰缺得稍多），但缺值明顯**集中在批次 7–8**。這比較像「某些批次的偵測極限變差」（和 Do 2018 的情況類似），既不是純粹的 MNAR，也不是完全隨機。

**決定（主流程）**：在 log2 尺度用 **KNN**（k = 5）補值——利用最相似的樣本來估計。理由：缺值和強度的關聯弱；而且這份資料每個生物樣本都有重複測量，相似樣本的資訊很有用。**這是一個判斷，不是唯一答案**；延伸 3E 用小實驗比較不同方法，Part 7 再做敏感度比較。

> ⚠️ 缺值機制**不能只靠資料診斷**來確定（看不到的值本來就看不到）。務必做敏感度比較：換一種補值法，結論會不會變？

In [ ]:
from sklearn.impute import KNNImputer

K_NEIGHBORS = 5
L_f = np.log2(X_f)                                     # 在 log 尺度補值（強度跨好幾個數量級）
L_imp = pd.DataFrame(KNNImputer(n_neighbors=K_NEIGHBORS).fit_transform(L_f),
                     index=L_f.index, columns=L_f.columns)
X_imp = 2 ** L_imp                                     # 回到原始強度尺度，給後面的比例型校正用
n_imputed = int(X_f.isna().values.sum())
print(f"補了 {n_imputed} 個值（{X_f.isna().values.mean():.1%}）；剩餘缺值：{int(X_imp.isna().values.sum())}")

## 3E. 延伸（教師示範）：補值法比較——人為遮蔽再還原

**想法**（縮小版的 Kokla 2019 思路，**不是**重現該論文）：從「完全沒有缺值」的峰中隨機取 150 個，**故意遮掉**一些已知的值，用不同方法補回來，再和真值比較。誤差用 NRMSE = √(平均平方誤差) ÷ 真值的標準差（越小越好）。

兩種遮蔽情境：
- **MCAR**：隨機遮掉 5% 的值。
- **左截尾 MNAR**：每個峰最低的 5% 值全部遮掉（模擬低於偵測極限）。

比較方法：最小值、最小值的一半（min/2）、KNN、random forest（sklearn `IterativeImputer` ＋ `RandomForestRegressor`，類似 missForest 的做法；約 30 秒）。

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401（啟用實驗性功能）
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # RF 只跑 3 輪，示範用

rng = np.random.default_rng(42)
complete = L_f.loc[:, L_f.notna().all()]
L_true = complete.sample(n=150, axis=1, random_state=42)

def nrmse(true, imputed, mask):
    return np.sqrt(np.mean((true[mask] - imputed[mask]) ** 2) / np.var(true[mask]))

IMPUTERS = {
    "min":    lambda L: L.fillna(L.min()),
    "min/2":  lambda L: L.fillna(L.min() - 1),          # log2 尺度：一半 = 減 1
    "KNN":    lambda L: pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(L), index=L.index, columns=L.columns),
    "RF":     lambda L: pd.DataFrame(IterativeImputer(
                  estimator=RandomForestRegressor(n_estimators=30, max_depth=8, n_jobs=-1, random_state=0),
                  max_iter=3, n_nearest_features=20, random_state=0).fit_transform(L),
                  index=L.index, columns=L.columns),
}
MASKS = {
    "MCAR (random 5%)":           rng.random(L_true.shape) < 0.05,
    "MNAR (lowest 5% per peak)":  (L_true < L_true.quantile(0.05)).values,
}
rows = []
for mname, mask in MASKS.items():
    L_masked = L_true.mask(mask)
    for iname, f in IMPUTERS.items():
        t = time.time()
        rows.append({"scenario": mname, "method": iname,
                     "NRMSE": nrmse(L_true.values, f(L_masked).values, mask), "seconds": time.time() - t})
impute_bench = pd.DataFrame(rows)
display(impute_bench.pivot(index="method", columns="scenario", values="NRMSE").round(3))

**看結果、想一想**
- 隨機缺值時，KNN／RF 遠勝最小值類方法；左截尾時反過來，「最小值」最好。這和 Kokla 2019 的整體結論方向一致。
- 注意 **min/2 在左截尾情境也輸給「min」**：真值只比觀察到的最小值低一點點，「砍一半」補得太低了。常用的經驗法則不一定適合你的資料。
- 這個實驗只用了 150 個峰、一次隨機種子，而且模擬的機制是我們自己設定的——真實缺值機制未知，所以它只能告訴你「各方法在哪種情境下比較好」，不能替你決定資料的機制。

## Part 4. 訊號漂移：QC-LOESS 校正（投影片 ⑦）

**為什麼**：連續注入時，離子源逐漸變髒、儀器狀態改變，訊號會隨注入序慢慢漂移；不同天（批次）的整體強度也不同。QC 是同一管樣本，它的變化＝儀器的變化（Dunn 2011 的 QC-LOESS／QC-RSC 思路）。

**怎麼做**（每個峰、每個批次分開）：
1. 用該批次的 QC 強度對注入序做 LOESS（`statsmodels` 的 `lowess`）。
2. 把曲線內插到該批次所有樣本的注入序 → 得到每次注入的「儀器狀態」。
3. 校正值 = 原始值 ÷ 曲線 × 該峰所有 QC 的中位數。最後一步同時把各批次拉到同一個水準。

函式已寫好；你要調的是參數 `FRAC`（LOESS 每次局部擬合用多少比例的點；每批次只有 4–7 個 QC，太小會不穩）。

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess

def qc_loess_correct(D, meta, frac=1.0):
    """D：原始強度尺度、沒有缺值的矩陣。回傳 (校正後矩陣, 每次注入的擬合曲線值)"""
    out = D.copy(); trend = D.copy() * np.nan
    qc_mask = (meta["Class"] == "QC").values
    target = D[qc_mask].median()                       # 每個峰的 QC 中位數（校正目標）
    for b in sorted(meta["Batch"].unique()):
        in_b = (meta["Batch"] == b).values
        order_b = meta["Order"].values[in_b]
        order_qc = meta["Order"].values[in_b & qc_mask]
        for col in D.columns:
            y = D.loc[in_b & qc_mask, col].values
            fit = lowess(y, order_qc, frac=frac, it=0, return_sorted=True)
            f = np.interp(order_b, fit[:, 0], fit[:, 1])   # 批次頭尾以外：沿用端點值
            f = np.where(f > 0, f, np.median(y))            # 防呆：曲線不可為 0 或負
            trend.loc[in_b, col] = f
            out.loc[in_b, col] = D.loc[in_b, col].values / f * target[col]
    return out, trend

In [ ]:
FRAC = 1.0                                   # 試試 0.6、0.8、1.0
t = time.time()
X_corr, X_trend = qc_loess_correct(X_imp, meta, frac=FRAC)
print(f"QC-LOESS 完成，{X_imp.shape[1]} 個峰，{time.time()-t:.1f} 秒")

rsd_imp = X_imp[is_qc].std() / X_imp[is_qc].mean() * 100
rsd_corr = X_corr[is_qc].std() / X_corr[is_qc].mean() * 100
print(f"QC RSD 中位數：校正前 {rsd_imp.median():.1f}% → 校正後 {rsd_corr.median():.1f}%")

# 挑一個「漂移明顯」的峰來看：高強度峰中，校正後 QC RSD 下降最多者
strong = X_imp.mean() > X_imp.mean().median()
FEATURE = (rsd_imp - rsd_corr)[strong].idxmax()
print("示範的峰：", FEATURE)

In [ ]:
def plot_drift(feature):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
    for ax, D, ttl in [(axes[0], X_imp, "Before"), (axes[1], X_corr, "After QC-LOESS")]:
        for b in sorted(meta["Batch"].unique()):
            m = (meta["Batch"] == b).values
            ax.scatter(meta.loc[m & ~is_qc, "Order"], D.loc[m & ~is_qc, feature], s=18, color=BATCH_COLORS[b-1])
            if ttl == "Before":
                ax.plot(meta.loc[m, "Order"], X_trend.loc[m, feature], color="k", lw=1.5)
        ax.scatter(meta.loc[is_qc, "Order"], D.loc[is_qc, feature], s=80, marker="*", color="#EE6C4D",
                   edgecolor="k", label="QC", zorder=3)
        ax.set_xlabel("Injection order"); ax.set_title(f"{ttl}: {feature}"); ax.legend()
    axes[0].set_ylabel("Intensity (a.u.)")
    plt.tight_layout(); plt.show()

plot_drift(FEATURE)

**看結果**：左圖黑線是每個批次的 QC-LOESS 曲線；校正後（右圖）QC 被拉到同一水準，生物樣本（彩色點）也跟著被拉平。

**想一想**
1. 把 `FRAC` 改成 0.6 再跑一次，QC RSD 中位數變了多少？為什麼「QC 的 RSD 變小」不一定代表校正變好了？（提示：曲線就是用 QC 擬合的。）
2. 每批次只有 4–7 個 QC。如果某個批次的 QC 很少，LOESS 曲線會怎樣？
3. 換一個峰畫：`plot_drift(X_imp.columns[0])`。

## 4E. 延伸（教師示範）：不靠 QC 的批次校正——簡化版位置／尺度 vs ComBat

如果實驗**沒有 QC**，常見做法是直接用所有樣本估計批次效應。

- **簡化版位置／尺度校正**（我們自己寫）：在 log2 尺度，每個峰、每個批次，把平均與標準差調成和全部樣本一樣：
  `z = (x − 批次平均) ÷ 批次標準差 × 全體標準差 + 全體平均`
- **ComBat**（Johnson 2007）做的事類似，但多了**經驗貝氏（empirical Bayes）收縮**：每個峰的批次平均／變異數估計會「借用」所有峰的資訊往共同分布拉近，所以批次樣本數少時更穩健；也可以放入生物共變量（`mod`）來保護組別差異。我們的簡化版**沒有收縮、沒有共變量**。

⚠️ 兩者都假設**批次與組別沒有混淆**——它們會把「批次之間的平均差異」全部當成技術誤差移除。若某批次全是病人、另一批次全是對照，組別差異也會被一起移除。這份資料是平衡設計（練習 1），所以可以拿來示範。

下面需要安裝 `combat`（Colab 上約數秒；只多裝 `mpmath`，不會更動 pandas／numpy 版本）。

In [ ]:
%pip install -q combat==0.3.3

In [ ]:
import contextlib, io
from combat.pycombat import pycombat

def batch_location_scale(L, batch):
    out = L.copy()
    for b in np.unique(batch):
        m = (batch == b)
        out.loc[m] = (L.loc[m] - L.loc[m].mean()) / L.loc[m].std() * L.std() + L.mean()
    return out

batch = meta["Batch"].values
L_ls = batch_location_scale(L_imp, batch)
with contextlib.redirect_stdout(io.StringIO()):          # pycombat 會印很多訊息，先關掉
    L_combat = pycombat(L_imp.T, list(batch)).T          # pycombat 要「特徵 × 樣本」

L_qcloess = np.log2(X_corr)
batch_compare = {}
for name, L in [("KNN imputed (no correction)", L_imp), ("QC-LOESS (main flow)", L_qcloess),
                ("Location/scale (no QC)", L_ls), ("ComBat (no QC)", L_combat)]:
    batch_compare[name] = pca_metrics(run_pca(L)[0])
batch_compare = pd.DataFrame(batch_compare).T
batch_compare.round(3)

**看結果、想一想**
- 不靠 QC 的兩種方法都把「批次對 PC 的 R²」壓到接近 0——**這是定義上必然的**（它們就是把批次平均拉齊），所以這個指標不能拿來證明它們比較好。
- 比較有說服力的是**獨立的檢查**：同一生物樣本的重複測量有沒有變一致（Replicate dispersion）、物種差異（R² class）有沒有被保留。
- 想一想：如果某批次全是牛，`batch_location_scale` 會把牛和羊的差異怎麼了？

## Part 5. 樣本層級正規化：PQN（投影片 ⑦）

**為什麼**：每個樣本的總量（組織萃取的濃度、稀釋程度）不同，會讓**一個樣本所有峰一起變大或變小**。這和「某個峰隨注入序漂移」是不同的問題。

**怎麼做**（Dieterle 2006 的機率商數正規化，PQN）：
1. 參考圖譜 = 每個峰在 QC 的中位數。
2. 每個樣本的每個峰除以參考 → 商數。
3. 該樣本的**稀釋因子 = 所有商數的中位數**（用中位數，少數真的有變化的峰不會影響它）。
4. 樣本除以自己的稀釋因子。

### ✏️ 練習 4：算出每個樣本的稀釋因子

In [ ]:
reference = X_corr[is_qc].median()                 # 參考圖譜（每個峰一個值）
quotients = X_corr.div(reference, axis=1)          # 每個樣本、每個峰的商數
dilution = ____                                    # TODO：每個樣本（每一列）商數的中位數

<details><summary>💡 提示（先自己試，再打開）</summary>

「每一列」是 `axis=1`。`quotients.median(axis=...)`。

</details>

> 🆘 **卡住時**：① 先讀錯誤訊息最後一行（哪個變數、哪一行）→ ② 問 Colab 的 Gemini（把錯誤訊息貼給它，請它「解釋」而不是「直接改」）→ ③ 打開下面的提示。

In [ ]:
X_pqn = X_corr.div(dilution, axis=0)
print(dilution.describe().round(3))
fig, ax = plt.subplots(figsize=(9, 3.5))
sns.boxplot(x=meta["Batch"], y=dilution, hue=meta["Class"], palette=colors, ax=ax)
ax.axhline(1, color="k", lw=0.8); ax.set_ylabel("PQN dilution factor"); ax.set_xlabel("Batch")
plt.show()

**想一想**：稀釋因子大多在 0.9–1.1；有沒有哪個樣本特別偏離？如果某一組（例如病人）系統性地稀釋因子比較大，PQN 會不會把真正的生物差異也「正規化」掉？

## Part 6. 轉換與縮放（投影片 ⑧）

**為什麼**（van den Berg 2006）：
- 強度跨好幾個數量級，而且誤差大致和強度成比例 → **log2 轉換**讓分布較對稱、倍數變化變成加減。（0 或缺值不能取 log；我們已經補值了。glog 是另一種能處理接近 0 的值的轉換。）
- **mean-centering**：每個峰減去平均，讓 PCA 看的是「樣本間的差異」。
- **縮放**決定每個峰在 PCA 裡的「發言權」：
  - 不縮放：高變異的峰主導。
  - **auto scaling**（÷ 標準差）：每個峰變異都＝1，雜訊大的小峰也被放大。
  - **Pareto scaling**（÷ √標準差）：折衷，代謝體學很常用。

### ✏️ 練習 5：完成 Pareto scaling

In [ ]:
L_final = np.log2(X_pqn)
centered = L_final - L_final.mean()
auto = centered / L_final.std()
pareto = ____            # TODO：centered 除以「標準差的平方根」

<details><summary>💡 提示（先自己試，再打開）</summary>

和上一行 `auto` 幾乎一樣，只是分母改成 `np.sqrt(...)`。

</details>

> 🆘 **卡住時**：① 先讀錯誤訊息最後一行（哪個變數、哪一行）→ ② 問 Colab 的 Gemini（把錯誤訊息貼給它，請它「解釋」而不是「直接改」）→ ③ 打開下面的提示。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, (name, M) in zip(axes, [("centered only", centered), ("auto scaling", auto), ("Pareto scaling", pareto)]):
    ax.scatter(L_final.mean(), M.std(), s=5, alpha=0.4, color="#0F8B8D")
    ax.set_xlabel("log2 mean intensity"); ax.set_ylabel("SD after scaling"); ax.set_title(name)
plt.tight_layout(); plt.show()

**看結果**：auto scaling 之後每個峰的標準差都是 1（一條水平線）；Pareto 保留了部分原本的差異，但壓縮了範圍。

**想一想**：如果有一個峰幾乎只有雜訊，在哪一種縮放下它對 PCA 的影響最大？

## Part 7. 前處理後的 PCA 與量化指標

用和 Part 1**完全相同**的 `run_pca`（centering ＋ Pareto）與 `pca_metrics`，比較前後。

In [ ]:
scores_after, var_after = run_pca(L_final)
plot_pca(scores_after, var_after, "After")
metrics_after = pca_metrics(scores_after)
compare = pd.DataFrame({"Before (all peaks)": metrics_before, "After (main flow)": metrics_after})
compare.round(3)

### 解讀時一定要記得
1. **QC 聚攏不等於校正正確**：QC-LOESS 和 PQN 都是「用 QC 來定義正確」，QC 變整齊在某種程度上是必然的（循環論證）。要同時看**不參與校正的資訊**：這份資料可以看同一生物樣本的重複測量（Replicate dispersion）。
2. **要同時看 QC、批次、組別三件事**：QC 有沒有聚攏、批次的影響有沒有變小、**組別差異（牛 vs 羊）有沒有被保留**。如果組別的 R² 也大幅下降，可能是過度校正。
3. **批次與組別若混淆，事後校正無法補救**：數學上分不開的東西，再好的演算法也分不開。只能靠實驗設計（隨機化、每批次平衡放入各組與 QC）預防。
4. 下面的敏感度比較：換一種補值法、或改成「先校正再用 RSD 過濾」，結果會差多少？

In [ ]:
def main_flow(X, impute="knn", rsd_max=RSD_MAX, filter_after=False):
    """把主流程包成一個函式，方便做敏感度比較。X：0 已換成 NaN 的原始矩陣。"""
    def rsd(D): return D[is_qc].std() / D[is_qc].mean() * 100
    if not filter_after:
        X = X.loc[:, rsd(X) <= rsd_max]
    L = np.log2(X)
    if impute == "knn":
        L = pd.DataFrame(KNNImputer(n_neighbors=K_NEIGHBORS).fit_transform(L), index=L.index, columns=L.columns)
    else:                                                       # min/2
        L = L.fillna(L.min() - 1)
    Xc, _ = qc_loess_correct(2 ** L, meta, frac=FRAC)
    if filter_after:
        Xc = Xc.loc[:, rsd(Xc) <= rsd_max]
    ref = Xc[is_qc].median()
    Xp = Xc.div(Xc.div(ref, axis=1).median(axis=1), axis=0)
    return np.log2(Xp)

sens = {"KNN, filter first (main)": pd.concat([metrics_after, pd.Series({"n peaks": L_final.shape[1]})])}
for name, kw in [("min/2, filter first", dict(impute="min2")),
                 ("KNN, correct then filter", dict(filter_after=True))]:
    L_s = main_flow(X_all, **kw)
    sens[name] = pd.concat([pca_metrics(run_pca(L_s)[0]), pd.Series({"n peaks": L_s.shape[1]})])
pd.DataFrame(sens).round(3)

**想一想**：哪些指標對補值方法敏感、哪些不敏感？「先校正再過濾」多留下了多少峰？你會怎麼在報告中寫這些選擇？

## Part 8. 總結：每一步做了什麼、改了多少

前處理沒有唯一標準答案，**每一步的選擇與影響都要記錄下來**，讀者才能重現與評估。

In [ ]:
summary = pd.DataFrame([
    ["1 讀入", "刪除 'mean intensity' 列", raw.shape[0], raw.shape[1], f"{is_qc.sum()} QC、{(~is_qc).sum()} 生物樣本"],
    ["2 QC RSD 過濾", f"RSD ≤ {RSD_MAX}%", X_f.shape[0], X_f.shape[1], f"刪除 {raw.shape[1]-X_f.shape[1]} 個峰"],
    ["3 補值", f"KNN (k={K_NEIGHBORS}, log2)", X_imp.shape[0], X_imp.shape[1], f"補了 {n_imputed} 個值（{n_imputed/X_f.size:.1%}）"],
    ["4 QC-LOESS", f"每批次 LOESS, frac={FRAC}", X_corr.shape[0], X_corr.shape[1],
     f"QC RSD 中位數 {rsd_imp.median():.1f}% → {rsd_corr.median():.1f}%"],
    ["5 PQN", "QC 中位數為參考", X_pqn.shape[0], X_pqn.shape[1],
     f"稀釋因子 {dilution.min():.2f}–{dilution.max():.2f}"],
    ["6 轉換縮放", "log2 + Pareto", pareto.shape[0], pareto.shape[1], "可做統計／PCA 的矩陣"],
    ["7 PCA", "前 → 後", "", "",
     f"QC 離散度 {metrics_before['QC dispersion']:.2f}→{metrics_after['QC dispersion']:.2f}；"
     f"批次 R²(PC2) {metrics_before['R2 batch~PC2']:.2f}→{metrics_after['R2 batch~PC2']:.2f}"],
], columns=["步驟", "做法", "樣本數", "特徵數", "改變"])
summary

### 帶回家
- 主流程的每個參數（`RSD_MAX`、`K_NEIGHBORS`、`FRAC`、縮放方法）都是**選擇**，要記錄、要做敏感度比較。
- 評估校正要用**獨立**的證據（重複測量、已知標準品），不能只看 QC。
- 最重要的品質控制在**實驗設計**階段：隨機化、平衡批次、穿插 QC。
- 下一步：這個矩陣可以拿去做差異分析、生物標記篩選（下週）與代謝物鑑定。